<a href="https://colab.research.google.com/github/gamaraljaliy/Murshid_Gamar_Sadaia/blob/main/homework/day2_tools.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Day 2 Homework — Controlled Tool Calls

## Overview

This exercise implements a **read-only tool dispatcher** using fictional application records. It practices checking the structure of an untrusted tool call and verifying whether the current demo user may access a record. It is separate from the Day 2 core notebook.

**Workflow:** validate call structure → allowlisted tool → validate reference → check record ownership → return a limited response.

The tool call is a hand-written Python dictionary. **No LLM chooses or invokes tools**, and no external API, database, or real user account is used.

## Run

1. Use a Python 3 runtime in Google Colab.
2. Run the three code cells in order: sample records, `dispatch()`, and the seven-case test suite.
3. Compare the printed report with `results/homework/day2_tools_result.json` in the repository.

No third-party packages, authentication keys, or downloads are needed.

In [11]:

import json

RECORDS = {
    'REF-1001': {
        'owner': 'demo-user-1',
        'status': 'under_review'
    },
    'REF-1002': {
        'owner': 'demo-user-2',
        'status': 'approved'
    }
}

print('Setup ready')


Setup ready


In [12]:

def dispatch(call, trusted_user):

    # Validate the tool-call structure
    if not isinstance(call, dict) or set(call) != {'name', 'arguments'}:
        return {'code': 'invalid_call'}

    # Allow only the check_status tool
    if call['name'] != 'check_status':
        return {'code': 'tool_not_allowed'}

    # Validate arguments
    args = call['arguments']

    if not isinstance(args, dict) or set(args) != {'reference'}:
        return {'code': 'invalid_arguments'}

    ref = args['reference']

    if not isinstance(ref, str) or len(ref) != 8 or not ref.startswith('REF-') or not ref[4:].isascii() or not ref[4:].isdigit():
        return {'code': 'invalid_arguments'}

    # Check record ownership
    record = RECORDS.get(ref)

    if record is None or record['owner'] != trusted_user:
        return {'code': 'unavailable'}

    return {'code': 'ok', 'status': record['status']}


In [13]:

def run_tool_homework():
    before = json.dumps(RECORDS, sort_keys=True)

    cases = [
        ({'name': 'check_status', 'arguments': {'reference': 'REF-1001'}},
         {'code': 'ok', 'status': 'under_review'}),

        ({'name': 'check_status', 'arguments': {'reference': 'REF-1002'}},
         {'code': 'unavailable'}),

        ({'name': 'check_status', 'arguments': {'reference': 'REF-9999'}},
         {'code': 'unavailable'}),

        ({'name': 'delete_record', 'arguments': {'reference': 'REF-1001'}},
         {'code': 'tool_not_allowed'}),

        ({'name': 'check_status', 'arguments': {'reference': 'REF-1001', 'owner': 'demo-user-2'}},
         {'code': 'invalid_arguments'}),

        ({'name': 'check_status', 'arguments': {'reference': 1001}},
         {'code': 'invalid_arguments'}),

        ('not a tool object',
         {'code': 'invalid_call'})
    ]

    rows = []

    for index, (call, expected) in enumerate(cases, 1):
        actual = dispatch(call, trusted_user='demo-user-1')

        assert actual == expected, (index, actual, expected)

        rows.append({
            'case': index,
            'result': actual,
            'passed': actual == expected
        })

    assert json.dumps(RECORDS, sort_keys=True) == before

    result = {
        'mode': 'simulated_tool_protocol',
        'passed': len(rows),
        'total': len(cases),
        'records_unchanged': True,
        'cases': rows
    }

    print(json.dumps(result, indent=2))
    return result


result = run_tool_homework()


{
  "mode": "simulated_tool_protocol",
  "passed": 7,
  "total": 7,
  "records_unchanged": true,
  "cases": [
    {
      "case": 1,
      "result": {
        "code": "ok",
        "status": "under_review"
      },
      "passed": true
    },
    {
      "case": 2,
      "result": {
        "code": "unavailable"
      },
      "passed": true
    },
    {
      "case": 3,
      "result": {
        "code": "unavailable"
      },
      "passed": true
    },
    {
      "case": 4,
      "result": {
        "code": "tool_not_allowed"
      },
      "passed": true
    },
    {
      "case": 5,
      "result": {
        "code": "invalid_arguments"
      },
      "passed": true
    },
    {
      "case": 6,
      "result": {
        "code": "invalid_arguments"
      },
      "passed": true
    },
    {
      "case": 7,
      "result": {
        "code": "invalid_call"
      },
      "passed": true
    }
  ]
}


## Test Results and Design Notes

The saved run reports **7/7 passed**, `mode=simulated_tool_protocol`, and `records_unchanged=true`.

| Case | Input | Expected outcome |
| --- | --- | --- |
| 1 | Owned reference `REF-1001` | `ok`, `under_review` |
| 2 | Someone else's reference `REF-1002` | `unavailable` |
| 3 | Missing reference `REF-9999` | `unavailable` |
| 4 | Disallowed `delete_record` tool | `tool_not_allowed` |
| 5 | Unexpected `owner` argument | `invalid_arguments` |
| 6 | Numeric reference instead of text | `invalid_arguments` |
| 7 | Non-dictionary tool call | `invalid_call` |

### Validation vs. authorization

**Validation** checks that the call has exactly `name` and `arguments`, the tool is `check_status`, and the reference matches `REF-` followed by four ASCII digits. Invalid requests are rejected before record lookup.

**Authorization** checks the record owner using the separately supplied `trusted_user`, not an `owner` value inside untrusted arguments. A valid reference alone does not grant access. Missing and unauthorized records both return `unavailable` to avoid exposing the private status.

The tests also compare a snapshot of `RECORDS` before and after the suite, confirming that no records were modified.

### Limitations

This is a **simulated teaching exercise**. `trusted_user` is a fictional identity, not verified authentication; the records are in memory, not a real database. The implementation demonstrates call validation and ownership checks but not provider-native LLM function calling or production security.